# Sentence embeddings

In questo notebook passiamo dalle singole parole alle **frasi**, visualizziamo lo spazio
con t-SNE/UMAP, costruiamo un mini sistema di **semantic search** (il cuore del RAG).

**Modello:** `paraphrase-multilingual-MiniLM-L12-v2` (~420MB, gira su CPU, supporta italiano)

---

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
## Parte 1: Sentence Embeddings

A differenza di Word2Vec/GloVe (un vettore per parola), i sentence transformers producono
un **singolo vettore per un'intera frase**. Usano un transformer (BERT-like) con mean pooling.

Vantaggio: catturano il **significato complessivo**, non solo le parole individuali.

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("sentence-model")
model = AutoModel.from_pretrained("sentence-model")
model.eval()


In [ ]:
import torch
import torch.nn.functional as F

texts = [
    "Ciao, come stai?",
    "Hello, how are you?",
]

tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
display(tokens)

outputs = model(**tokens)
display(outputs.last_hidden_state.shape)

mask = tokens["attention_mask"].unsqueeze(-1).float()                           # just needed for matrix multiplication
embeddings = (outputs.last_hidden_state * mask).sum(axis=1) / mask.sum(axis=1)  # (batch_size, hidden_size)
embeddings = F.normalize(embeddings, p=2, dim=1)
display(embeddings.shape)


In [ ]:
# Classe helper per sentence embeddings
class SentenceEncoder:
    """Sentence embeddings via mean pooling + L2 norm."""
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def encode(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            out = model(**inputs)
        # Mean pooling: media pesata sui token non-padding
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        emb = (out.last_hidden_state * mask).sum(axis=1) / mask.sum(axis=1)
        emb = F.normalize(emb, p=2, dim=1)
        return emb
    
encoder = SentenceEncoder(tokenizer, model)


In [ ]:
# Similarita tra frasi
sentences = [
    "Il gatto dorme sul divano",
    "Un felino riposa sul sofa",
    "The cat sleeps on the couch",
    "Il cane gioca nel giardino",
    "La borsa e in calo oggi",
    "Die Katze schlaft auf dem Sofa",
]

embeddings = encoder.encode(sentences)
sim_matrix = (embeddings @ embeddings.T).numpy() # cosine similarity tra tutte le coppie di frasi (dot product tra vettori normalizzati)

fig = px.imshow(
    sim_matrix,
    x=sentences, y=sentences,
    color_continuous_scale="RdBu_r",
    zmin=-0.2, zmax=1.0,
    title="Matrice di similarita tra frasi (cosine similarity)",
    width=800, height=600,
)
fig.update_layout(template="plotly_white")
fig.show()


In [ ]:
# Corpus tematico per la visualizzazione
corpus = {
    "Scienza": [
        "L'energia non si crea ne si distrugge, si trasforma",
        "Il DNA contiene le istruzioni genetiche di ogni organismo",
        "I buchi neri deformano lo spazio-tempo",
        "La fotosintesi converte luce solare in energia chimica",
        "I quanti possono essere in sovrapposizione di stati",
    ],
    "Sport": [
        "La squadra ha vinto il campionato con un gol all'ultimo minuto",
        "Il tennista ha servito tre ace nel tie-break",
        "La maratona richiede mesi di preparazione fisica",
        "Il nuotatore ha battuto il record mondiale nei 100 stile",
        "La partita di basket e finita ai supplementari",
    ],
    "Cucina": [
        "La carbonara si fa con guanciale, uova, pecorino e pepe",
        "Il sushi richiede riso preparato con aceto di riso",
        "La lievitazione dell'impasto dura almeno 24 ore",
        "Il brodo di carne si prepara con sedano, carota e cipolla",
        "La temperatura del forno deve essere di 220 gradi",
    ],
    "Tecnologia": [
        "Il machine learning impara pattern dai dati",
        "I container Docker isolano le applicazioni",
        "Il protocollo HTTPS cifra la comunicazione tra client e server",
        "Le reti neurali sono ispirate al cervello biologico",
        "Git permette il version control distribuito del codice",
    ],
    "Filosofia": [
        "Cogito ergo sum: il pensiero prova l'esistenza",
        "L'etica kantiana si basa sull'imperativo categorico",
        "Il mito della caverna di Platone descrive la conoscenza",
        "Nietzsche ha dichiarato la morte di Dio",
        "Il libero arbitrio e un'illusione secondo i deterministi",
    ],
}

all_sentences = []
all_categories = []
for cat, sents in corpus.items():
    all_sentences.extend(sents)
    all_categories.extend([cat] * len(sents))

corpus_embeddings = encoder.encode(all_sentences)
print(f"Embeddings shape: {corpus_embeddings.shape}")

---
## Parte 2: Visualizzare lo spazio

Usiamo **t-SNE** (t-distributed Stochastic Neighbor Embedding) per proiettare in 2D.

t-SNE preserva la **struttura locale** (chi e vicino a chi) ma le distanze globali.
Il parametro `perplexity` controlla quanti vicini considerare.


In [ ]:
# t-SNE: come la perplexity cambia il risultato
fig = make_subplots(rows=1, cols=3, subplot_titles=["Perplexity=5", "Perplexity=10", "Perplexity=20"])

colors_map = {cat: px.colors.qualitative.Set2[i] for i, cat in enumerate(corpus.keys())}

for col, perp in enumerate([5, 10, 20], 1):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1000)
    coords = tsne.fit_transform(corpus_embeddings)

    for cat in corpus.keys():
        mask = [c == cat for c in all_categories]
        cat_coords = coords[mask]
        cat_sents = [s for s, m in zip(all_sentences, mask) if m]
        fig.add_trace(go.Scatter(
            x=cat_coords[:, 0], y=cat_coords[:, 1],
            mode="markers",
            hovertext=cat_sents,
            hoverinfo="text",
            name=cat,
            marker=dict(color=colors_map[cat], size=10),
            showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(
    title="t-SNE: la perplexity cambia il layout",
    width=1200, height=500, template="plotly_white",
)
fig.show()

print("ATTENZIONE: le distanze tra cluster in t-SNE sono ARBITRARIE.")
print("Solo la struttura LOCALE (chi e vicino a chi) e affidabile.")


Con t-SNE, se vedi il cluster "Scienza" lontano dal cluster "Sport" nel plot, non puoi concludere che siano davvero lontani nello spazio originale. 

Un'alternativa è **UMAP** (Uniform Manifold Approximation and Projection) che preserva meglio la distanza globale. 

In pratica: in t-SNE puoi fidarti solo di "questi punti sono vicini tra loro". In UMAP puoi fidarti anche di "questo gruppo è più vicino a quello che
a quell'altro".

In [ ]:
import umap
# UMAP: preserva meglio la struttura globale rispetto a t-SNE

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=20, min_dist=0.1)
umap_coords = reducer.fit_transform(corpus_embeddings)

fig = px.scatter(
    x=umap_coords[:, 0], y=umap_coords[:, 1],
    color=all_categories,
    hover_name=all_sentences,
    title="UMAP - struttura globale meglio preservata",
    width=850, height=600,
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(template="plotly_white")
fig.show()


---
## Parte 3: Mini-RAG (Semantic Search)

RAG (Retrieval-Augmented Generation) funziona cosi:
1. **Indicizza**: calcola gli embeddings di tutti i documenti
2. **Query**: calcola l'embedding della domanda
3. **Retrieval**: trova i documenti piu simili (cosine similarity)
4. **Generation**: passa i documenti trovati all'LLM come contesto

Noi facciamo i passi 1-3. 

In [ ]:
# Il nostro "database" di conoscenza
knowledge_base = [
    "Python e un linguaggio di programmazione interpretato creato da Guido van Rossum nel 1991",
    "La Torre Eiffel e alta 330 metri e si trova a Parigi",
    "Il machine learning e un sottoinsieme dell'intelligenza artificiale",
    "La pizza margherita e nata a Napoli in onore della Regina Margherita",
    "Le reti neurali convoluzionali sono usate per il riconoscimento di immagini",
    "Roma e la capitale d'Italia e ha circa 2.8 milioni di abitanti",
    "I transformer hanno rivoluzionato il NLP con il meccanismo di attention",
    "Il Colosseo poteva contenere fino a 80.000 spettatori",
    "GPT sta per Generative Pre-trained Transformer",
    "La carbonara tradizionale non contiene panna",
    "BERT e un modello bidirezionale per la comprensione del linguaggio",
    "Il tiramisù e un dolce italiano a base di savoiardi e mascarpone",
    "L'algoritmo di backpropagation calcola i gradienti per l'addestramento",
    "Firenze e stata la culla del Rinascimento italiano",
    "I word embeddings mappano parole in vettori densi",
    "Il sushi e un piatto giapponese a base di riso e pesce crudo",
    "Tokyo e la capitale del Giappone con oltre 13 milioni di abitanti",
    "Il Big Ben e la famosa torre dell'orologio di Londra",
    "La Silicon Valley e il centro mondiale dell'innovazione tecnologica",
    "Dante Alighieri ha scritto la Divina Commedia nel XIV secolo",
]

# Indicizzazione
kb_embeddings = encoder.encode(knowledge_base)
print(f"Knowledge base indicizzata: {len(knowledge_base)} documenti")



In [ ]:
def semantic_search(query, top_k=3):
    """Cerca i documenti piu rilevanti per una query."""
    query_emb = encoder.encode(query)
    scores = query_emb @ kb_embeddings.T # per vettori normalizzati è la cosine similarity
    scores = scores.squeeze()
    top_indices = scores.argsort(descending=True)[:top_k]

    print(f"Query: \"{query}\"\n")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. [{scores[idx]:.4f}] {knowledge_base[idx]}")
    print()
    return [(knowledge_base[i], float(scores[i])) for i in top_indices]

# Proviamolo
semantic_search("Come funziona il deep learning?")
semantic_search("Cosa mangiare in Italia?")
semantic_search("Monumenti famosi")
print()



---
## Sfide


### Sfida: Aritmetica tra frasi

Nei word embeddings l'aritmetica funziona (king - man + woman = queen).
Funziona anche con le frasi? Proviamo con la knowledge base.

Costruisci un vettore query combinando embeddings di frasi e vedi cosa restituisce il RAG.
Esempio: prendi la frase sulla pizza, sottrai "Italia", aggiungi "Giappone". Quali sono i documenti più simili?


In [ ]:
# ============================================================
# SFIDA: Aritmetica tra frasi
# ============================================================


### Sfida: Filtro semantico

Il RAG base restituisce i documenti piu simili alla query.
Ma a volte vuoi cercare documenti su un argomento **escludendo** un altro.

Esempio: "cose sull'Italia ma non sul cibo" — come lo implementi?

Scrivi una funzione `semantic_search_filtrato` che accetti una query positiva
e una negativa, e restituisca solo documenti rilevanti al primo ma non al secondo.


In [ ]:
# ============================================================
# SFIDA: Implementa un filtro semantico per il RAG
# ============================================================

def semantic_search_filtrato(query_positiva, query_negativa, top_k=3, peso_negativo=0.5):
    """Cerca documenti rilevanti a query_positiva ma non a query_negativa."""
    # Il tuo codice qui
    pass

# Testa la tua funzione:
# semantic_search_filtrato("Italia", "cibo")
# Dovrebbe restituire Roma, Firenze, Dante... ma NON pizza, carbonara, tiramisù


### Sfida: I limiti del RAG

Il RAG cerca per **similarita tra frasi**, non capisce il **contenuto**.
Questo è un problema serio quando le frasi sono strutturalmente uguali ma differiscono nei dati.

Un LLM con tutti i documenti nel prompt saprebbe rispondere.
Il RAG, che seleziona per similarita, non ha modo di distinguerli.

Prova le seguenti cose:
* visualizza la similarità tra i documenti
* Fai domande tipo "Quale prodotto ha il fatturato piu alto?" oppure "Come è andato il prodotto Alpha nel secondo trimestre?"

In [ ]:
# Scenario: database aziendale di report trimestrali
kb_aziendale = [
    "Il prodotto Alpha ha generato un fatturato di 2.3 milioni di euro nel Q1 2024",
    "Il prodotto Beta ha generato un fatturato di 8.7 milioni di euro nel Q1 2024",
    "Il prodotto Gamma ha generato un fatturato di 1.1 milioni di euro nel Q1 2024",
    "Il prodotto Delta ha generato un fatturato di 15.2 milioni di euro nel Q1 2024",
    "Il prodotto Alpha ha generato un fatturato di 3.1 milioni di euro nel Q2 2024",
    "Il prodotto Beta ha generato un fatturato di 7.9 milioni di euro nel Q2 2024",
]


Esplora la similarità tra i documenti in cui ci sono frasi di significato opposto.

In [ ]:
# Frasi OPPOSTE nel significato
frasi_opposte = [
    "Il titolo in borsa e salito del 5% questa settimana",
    "Il titolo in borsa e sceso del 5% questa settimana",
    "Il paziente e migliorato dopo il trattamento",
    "Il paziente e peggiorato dopo il trattamento",
    "Il progetto e stato approvato dal consiglio",
    "Il progetto e stato bocciato dal consiglio",
]
